# PPO（Proximal Policy Optimization）の動作原理を数式レベルで確認する

<a href="https://colab.research.google.com/github/miyoshi-nii/llm_test/blob/main/docs/ppo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PPO（Schulman et al., 2017, *Proximal Policy Optimization Algorithms*）は、方策勾配法の一種で、**1 回集めたデータで方策を何度も更新しつつ、方策が一度に変わりすぎないようにする** 工夫（クリップ）を持つ強化学習アルゴリズムです。RLHF（InstructGPT など）で、報酬モデルの点数が高くなるように言語モデルを最適化する段階に使われてきました。

このノートでは

- **第 1〜4 節（NumPy）**：方策勾配とベースライン、アドバンテージと GAE、重要度サンプリングによる代理目的関数、PPO のクリップ目的関数を、数式と数値で 1 つずつ確認します。小さな環境では期待収益を **動的計画法で厳密に計算** できるので、近似がどこで成り立ち、どこで崩れるかを正確に見られます。
- **第 5 節（NumPy）**：小さな環境で PPO を実装し、クリップあり・なし・通常の方策勾配を比べます。
- **第 6 節（PyTorch）**：RLHF と同じ形で GPT-2 small を PPO で学習します（LoRA の方策 + 価値ヘッド + 参照方策との KL 罰則）。

DPO（PPO を使わずに好みデータから直接学習する方法）は [dpo.ipynb](dpo.ipynb) で扱いました。PPO は学習アルゴリズムなので、ごく小さな学習を含みます。

### 記号

| 記号 | 意味 |
|---|---|
| $s_t,\ a_t,\ r_t$ | 時刻 $t$ の状態・行動・報酬 |
| $\pi_\theta(a\mid s)$ | 方策（パラメータ $\theta$） |
| $\gamma$ | 割引率 |
| $G_t = \sum_{k\ge 0}\gamma^k r_{t+k}$ | 収益（return） |
| $V^\pi(s),\ Q^\pi(s,a)$ | 状態価値・行動価値 |
| $A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$ | アドバンテージ（平均よりどれだけ良い行動か） |
| $\rho_t = \pi_\theta(a_t\mid s_t)/\pi_{\theta_{\text{old}}}(a_t\mid s_t)$ | 重要度比（新旧の方策の確率比） |

### 目次
0. 準備
1. 方策勾配（REINFORCE）とベースライン
2. アドバンテージの推定：GAE
3. 重要度サンプリングと代理目的関数
4. PPO のクリップ目的関数
5. 小さな環境で PPO を動かす
6. RLHF：GPT-2 small を PPO で学習する
7. まとめと参考文献

## 0. 準備

### Google Colab で実行する場合
- 上の **Open in Colab** バッジから開くか、「ファイル → ノートブックをアップロード」で `ppo.ipynb` を選びます。
- 必要なライブラリ（`numpy`, `matplotlib`, `torch`, `transformers`）は Colab に入っています。第 6 節は文章の生成と学習を繰り返すので、GPU ランタイムを推奨します（CPU でも動きますが、10 分以上かかることがあります）。

### ローカル（このリポジトリの `.venv`）で実行する場合
```bash
uv pip install numpy matplotlib torch transformers ipykernel
```

In [ ]:
import math
import re
import sys
import time
from contextlib import contextmanager

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカル'} / Python {sys.version.split()[0]}"
      f" / numpy {np.__version__} / matplotlib {matplotlib.__version__}")


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

## 1. 方策勾配（REINFORCE）とベースライン

期待収益 $J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}[G_0]$ を最大化したいとき、**方策勾配定理** により

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\Big[\nabla_\theta\log\pi_\theta(a_t\mid s_t)\,\big(G_t - b(s_t)\big)\Big]
$$

と書けます（REINFORCE）。$b(s)$ は **ベースライン** で、行動に依存しなければ何を引いても期待値は変わりません（$\mathbb{E}_{a\sim\pi}[\nabla\log\pi(a\mid s)] = \nabla\sum_a\pi(a\mid s) = \nabla 1 = 0$ だから）。一方で **分散は大きく変わります**。$b(s) = V^\pi(s)$ とすると $G_t - V(s_t)$ はアドバンテージの推定になります。

1 状態だけの問題（多腕バンディット）で確かめます。5 本の腕の平均報酬が $\mu = (1.0, 0.8, 0.5, 0.2, 0.0)$、報酬には標準偏差 1 のノイズが乗り、方策は $\pi_\theta = \mathrm{softmax}(\theta)$ です。厳密な勾配は $\nabla_\theta J = \pi\odot(\mu - J)$（$J = \pi^\top\mu$）で、サンプルからの推定は $\frac{1}{N}\sum_i (r_i - b)(e_{a_i} - \pi)$ です（$\nabla_\theta\log\pi(a) = e_a - \pi$）。

In [ ]:
rng = np.random.default_rng(0)
mu = np.array([1.0, 0.8, 0.5, 0.2, 0.0])
theta = np.array([0.2, -0.1, 0.3, 0.0, 0.1])
pi = softmax(theta)
J_exact = pi @ mu
grad_exact = pi * (mu - J_exact)


def reinforce_estimates(shift, baseline, n_samples=32, n_repeat=20000):
    """報酬に shift を足した問題で、REINFORCE の勾配推定を n_repeat 回くり返す"""
    a = rng.choice(len(mu), p=pi, size=(n_repeat, n_samples))
    r = mu[a] + shift + rng.normal(size=a.shape)
    b = (J_exact + shift) if baseline else 0.0
    score = np.eye(len(mu))[a] - pi                          # ∇log π(a) = e_a - π
    return ((r - b)[..., None] * score).mean(axis=1)         # (n_repeat, 5)


print("厳密な勾配:", grad_exact)
print(f"{'報酬のずれ':>8} {'ベースライン':>10} | {'推定の平均との差':>14} | 分散（全成分の和）")
for shift in [0.0, 10.0]:
    for baseline in [False, True]:
        g = reinforce_estimates(shift, baseline)
        print(f"{shift:>10} {str(baseline):>12} | {np.abs(g.mean(0) - grad_exact).max():>14.4f} | {g.var(0).sum():.4f}")
print("→ どの設定でも平均は厳密な勾配に一致（不偏）。ベースラインを引くと分散が大きく下がり、報酬に定数が足されても影響を受けない")

## 2. アドバンテージの推定：GAE

アドバンテージ $A(s_t, a_t)$ を推定する方法には、偏りと分散のトレードオフがあります。価値関数の推定 $V$ を使った **TD 誤差** を

$$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

とすると、**GAE**（Generalized Advantage Estimation; Schulman et al., 2015）は

$$
\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty}(\gamma\lambda)^l\,\delta_{t+l} = \delta_t + \gamma\lambda\,\hat{A}_{t+1}
$$

です（右の漸化式で後ろから計算できる）。

- $\lambda = 0$：$\hat{A}_t = \delta_t$。分散は小さいが、$V$ が不正確だと偏る。
- $\lambda = 1$：$\hat{A}_t = \sum_l\gamma^l\delta_{t+l} = G_t - V(s_t)$（途中の $V$ が打ち消し合う）。偏りはないが分散が大きい（モンテカルロ収益）。

PPO の論文や多くの実装では $\lambda = 0.95$ 前後が使われます。終端の後の価値は 0 とします。

In [ ]:
def gae(rewards, values, gamma, lam):
    """values は長さ T+1（最後は終端後の価値 = 0）。後ろから漸化式で計算"""
    T = len(rewards)
    adv = np.zeros(T)
    last = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        last = delta + gamma * lam * last
        adv[t] = last
    return adv


rng = np.random.default_rng(1)
T, gamma, lam = 6, 0.9, 0.8
rewards = rng.normal(size=T)
values = np.append(rng.normal(size=T), 0.0)              # 終端の後の価値は 0
deltas = rewards + gamma * values[1:] - values[:-1]
explicit = np.array([sum((gamma * lam) ** l * deltas[t + l] for l in range(T - t)) for t in range(T)])
assert np.allclose(gae(rewards, values, gamma, lam), explicit)
print("✔ 漸化式 Â_t = δ_t + γλ Â_{t+1} は、定義 Σ (γλ)^l δ_{t+l} と一致")

assert np.allclose(gae(rewards, values, gamma, 0.0), deltas)
G = np.array([sum(gamma ** k * rewards[t + k] for k in range(T - t)) for t in range(T)])
assert np.allclose(gae(rewards, values, gamma, 1.0), G - values[:-1])
print("✔ λ = 0 なら TD 誤差 δ_t、λ = 1 なら G_t - V(s_t)")
print("λ = 0  :", gae(rewards, values, gamma, 0.0))
print("λ = 0.8:", gae(rewards, values, gamma, 0.8))
print("λ = 1  :", gae(rewards, values, gamma, 1.0))

## 3. 重要度サンプリングと代理目的関数

方策勾配法は、データを集めた方策 $\pi_{\theta_{\text{old}}}$ で 1 回勾配を計算したら、そのデータは捨てなければなりません（勾配の式が「今の方策でサンプルしたデータ」の期待値だから）。そこで **重要度サンプリング** で書き換えた **代理目的関数** を使うと、古い方策のデータで新しい方策を評価できます。

$$
L^{\text{CPI}}(\theta) = \mathbb{E}_{s,a\sim\pi_{\theta_{\text{old}}}}\Big[\rho_\theta(s,a)\,\hat{A}(s,a)\Big],\qquad \rho_\theta(s,a) = \frac{\pi_\theta(a\mid s)}{\pi_{\theta_{\text{old}}}(a\mid s)}
$$

- $\theta = \theta_{\text{old}}$ では $\nabla_\theta\rho = \nabla_\theta\log\pi_\theta$ なので、**$\nabla L^{\text{CPI}}$ は方策勾配と一致** します。
- しかし $L^{\text{CPI}}$ は「状態の分布は古い方策のまま」とみなした **1 次近似** です。厳密には（**性能差分補題**, performance difference lemma）

$$
J(\theta) - J(\theta_{\text{old}}) = \frac{1}{1-\gamma}\sum_s d^{\pi_\theta}(s)\sum_a\pi_\theta(a\mid s)\,A^{\pi_{\text{old}}}(s,a)
$$

で、状態の分布 $d^{\pi_\theta}$（割引付きの訪問分布）は **新しい方策** のものです。代理目的関数はこれを $d^{\pi_{\text{old}}}$ で置き換えたものなので、方策が大きく変わると近似が崩れます。TRPO の論文（Schulman et al., 2015）は、この誤差が新旧の方策の KL で抑えられること、すなわち

$$
J(\theta) - J(\theta_{\text{old}}) \ \ge\ L^{\text{CPI}}(\theta) - L^{\text{CPI}}(\theta_{\text{old}}) - C\cdot\max_s\mathrm{KL}\big(\pi_{\theta_{\text{old}}}(\cdot\mid s)\,\|\,\pi_\theta(\cdot\mid s)\big)
$$

（$C$ は定数）を示しました。KL が小さい範囲で代理目的関数を上げれば、真の性能も上がることが保証されます。これが「方策を一度に変えすぎてはいけない」理由で、TRPO（KL 制約）や PPO（クリップ）の動機です。

**小さな環境で厳密に確かめる**：9 マスの一本道（状態 0〜8）を考えます。エージェントはマス 2 から出発し、左右に 1 マス動けます。左端（0）に着くと報酬 0.2、右端（8）に着くと報酬 1.0 で終了します（$\gamma = 0.95$）。近くの小さな報酬と、遠くの大きな報酬があるわけです。状態と行動が少ないので、価値関数・訪問分布・期待収益を連立一次方程式で **厳密に** 計算できます。

In [ ]:
N_S, START, GAMMA = 9, 2, 0.95
TERMINAL = {0: 0.2, N_S - 1: 1.0}          # 終端マスとそこに着いたときの報酬
LEFT, RIGHT = 0, 1


def env_step(s, a):
    s2 = s - 1 if a == LEFT else s + 1
    return s2, TERMINAL.get(s2, 0.0), s2 in TERMINAL


def transition_matrix(pi):
    """方策 π のもとでの非終端状態間の遷移行列 P と、1 ステップの期待報酬 r̄"""
    P, rbar = np.zeros((N_S, N_S)), np.zeros(N_S)
    for s in range(1, N_S - 1):
        for a in (LEFT, RIGHT):
            s2, r, done = env_step(s, a)
            rbar[s] += pi[s, a] * r
            if not done:
                P[s, s2] += pi[s, a]
    return P, rbar


def exact_V(pi):
    P, rbar = transition_matrix(pi)
    return np.linalg.solve(np.eye(N_S) - GAMMA * P, rbar)           # V = (I - γP)^{-1} r̄


def exact_Q(pi, V):
    Q = np.zeros((N_S, 2))
    for s in range(1, N_S - 1):
        for a in (LEFT, RIGHT):
            s2, r, done = env_step(s, a)
            Q[s, a] = r + (0.0 if done else GAMMA * V[s2])
    return Q


def visitation(pi):
    """割引付き訪問分布 d(s) = (1-γ) Σ_t γ^t P(s_t = s)"""
    P, _ = transition_matrix(pi)
    return (1 - GAMMA) * np.linalg.solve((np.eye(N_S) - GAMMA * P).T, np.eye(N_S)[START])


def J_of(theta):
    return exact_V(softmax(theta))[START]


right_only = np.tile([0.0, 1.0], (N_S, 1))
left_only = np.tile([1.0, 0.0], (N_S, 1))
print(f"常に右: J = {exact_V(right_only)[START]:.4f}（= γ^5 = {GAMMA ** 5:.4f}）,  常に左: J = {exact_V(left_only)[START]:.4f}"
      f",  一様ランダム: J = {J_of(np.zeros((N_S, 2))):.4f}")

In [ ]:
theta_old = np.zeros((N_S, 2))                       # 一様ランダムな方策から出発
pi_old = softmax(theta_old)
V_old = exact_V(pi_old)
A_old = exact_Q(pi_old, V_old) - V_old[:, None]      # 厳密なアドバンテージ
d_old = visitation(pi_old)

# (1) 方策勾配定理：∂J/∂θ[s,a] = d(s) π(a|s) A(s,a) / (1-γ)  ←→ 数値微分
grad_pg = d_old[:, None] * pi_old * A_old / (1 - GAMMA)
eps = 1e-6
grad_num = np.zeros_like(theta_old)
for idx in np.ndindex(theta_old.shape):
    E = np.zeros_like(theta_old); E[idx] = eps
    grad_num[idx] = (J_of(theta_old + E) - J_of(theta_old - E)) / (2 * eps)
assert np.allclose(grad_pg, grad_num, atol=1e-8)
print("✔ 方策勾配定理の式は、厳密な J の数値微分と一致")


def surrogate(theta):
    """L(θ) - L(θ_old) = Σ_s d_old(s) Σ_a π_θ(a|s) A_old(s,a) / (1-γ)（状態分布は古い方策のまま）"""
    return np.sum(d_old[:, None] * softmax(theta) * A_old) / (1 - GAMMA)


def true_improvement_pdl(theta):
    """性能差分補題：状態分布を新しい方策のものにする"""
    pi = softmax(theta)
    return np.sum(visitation(pi)[:, None] * pi * A_old) / (1 - GAMMA)


rng = np.random.default_rng(0)
for _ in range(5):
    th = rng.normal(scale=2.0, size=theta_old.shape)
    assert np.isclose(J_of(th) - J_of(theta_old), true_improvement_pdl(th))
print("✔ 性能差分補題 J(θ) - J(θ_old) = Σ d^{π_θ} π_θ A^{π_old} / (1-γ) がランダムな方策で成立")

In [ ]:
g_dir = grad_pg / np.linalg.norm(grad_pg)
ts = np.linspace(0, 60, 121)
true_imp = [J_of(theta_old + t * g_dir) - J_of(theta_old) for t in ts]
surr_imp = [surrogate(theta_old + t * g_dir) for t in ts]
kls = [np.sum(d_old * np.sum(pi_old * (np.log(pi_old) - np.log(softmax(theta_old + t * g_dir))), axis=1)) for t in ts]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(ts, true_imp, label="true  J(θ) - J(θ_old)")
axes[0].plot(ts, surr_imp, "--", label="surrogate  L(θ) - L(θ_old)")
axes[0].set_xlabel("step size t along the policy gradient"); axes[0].set_title("true improvement vs. surrogate"); axes[0].legend()
ts_log = np.logspace(-3, np.log10(60), 80)                 # 小さな変化も見えるように対数間隔で
err = [abs(surrogate(theta_old + t * g_dir) - (J_of(theta_old + t * g_dir) - J_of(theta_old))) for t in ts_log]
kl_log = [np.sum(d_old * np.sum(pi_old * (np.log(pi_old) - np.log(softmax(theta_old + t * g_dir))), axis=1)) for t in ts_log]
axes[1].loglog(kl_log, err)
axes[1].set_xlabel("KL(π_old || π_θ) weighted by d_old"); axes[1].set_ylabel("|surrogate - true|")
axes[1].set_title("the surrogate error grows with the KL from the old policy")
plt.tight_layout(); plt.show()
for t in [0.1, 1, 5, 20, 60]:
    th = theta_old + t * g_dir
    ti, si = J_of(th) - J_of(theta_old), surrogate(th)
    klv = np.sum(d_old * np.sum(pi_old * (np.log(pi_old) - np.log(softmax(th))), axis=1))
    print(f"t = {t:>4}: 真の改善 {ti:.5f}, 代理目的関数 {si:.5f}, 相対誤差 {abs(si - ti) / ti:6.1%}, KL {klv:.4f}")

**観察**：
- 方策をわずかに動かしたとき（$t = 0.1$）は、代理目的関数と真の改善はほぼ一致します（勾配が一致するので 1 次まで同じ）。方策が離れるほど（KL が大きいほど）両者の差は大きくなります。右図の両対数グラフでは、誤差が **KL にほぼ比例する**（傾き 1 の直線）ことがわかります。TRPO の下界の「$C\cdot\mathrm{KL}$」と同じ形です。
- この環境では、代理目的関数は真の改善を **過小評価** しています。右へ進む確率を上げると右側の状態を訪れる割合が増え、そこでは右へ進むアドバンテージがさらに大きいので、古い状態分布で見積もった値より実際の改善が大きくなるためです。誤差の向きは問題によって変わり、逆に過大評価する（代理目的関数では良くなるはずなのに、実際には悪くなる）こともあります。
- 表形式で **厳密な** アドバンテージを使える場合は、代理目的関数を最大化する（各状態でアドバンテージ最大の行動を選ぶ）と必ず改善します（方策反復の方策改善定理）。実際の問題でクリップや KL 制約が必要になるのは、アドバンテージが **少ないサンプルからの推定値** で誤差を含むことと、関数近似ではある状態での更新が別の状態の行動まで変えてしまうことのためです。第 5 節でこれを見ます。

## 4. PPO のクリップ目的関数

PPO は、重要度比 $\rho_t$ が $[1-\epsilon, 1+\epsilon]$ の外に出ても得をしないように目的関数を「クリップ」します（$\epsilon = 0.2$ がよく使われる）。

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[\min\big(\rho_t\hat{A}_t,\ \mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\,\hat{A}_t\big)\Big]
$$

- $\hat{A}_t > 0$（良い行動）：$\rho_t$ を大きくすると得をするが、$\rho_t > 1+\epsilon$ では一定になり **勾配が 0**。良い行動の確率を一度に上げすぎない。
- $\hat{A}_t < 0$（悪い行動）：$\rho_t$ を小さくすると得をするが、$\rho_t < 1-\epsilon$ では一定になり勾配が 0。
- $\min$ をとるので、クリップしない値より大きくなることはありません（**悲観的な下界**）。たとえば $\hat{A}_t>0$ で $\rho_t < 1-\epsilon$ になってしまった場合（良い行動の確率を下げすぎた）は、クリップされずに元に戻す方向の勾配が流れます。

1 サンプルあたりの勾配は、表形式の方策なら

$$
\nabla_\theta\min(\cdots) = \begin{cases} 0 & (\hat{A}>0 \text{ かつ } \rho>1+\epsilon)\ \text{または}\ (\hat{A}<0 \text{ かつ } \rho<1-\epsilon) \\ \hat{A}\,\rho\,\nabla_\theta\log\pi_\theta(a\mid s) & \text{それ以外} \end{cases}
$$

In [ ]:
EPS = 0.2


def clip_obj(rho, A, eps=EPS):
    return np.minimum(rho * A, np.clip(rho, 1 - eps, 1 + eps) * A)


rhos = np.linspace(0, 2, 401)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, A in zip(axes, [1.0, -1.0]):
    ax.plot(rhos, rhos * A, ":", color="gray", label="unclipped  ρA")
    ax.plot(rhos, clip_obj(rhos, A), lw=2, label="L^CLIP")
    for v in (1 - EPS, 1 + EPS):
        ax.axvline(v, color="tab:red", ls="--", lw=0.8)
    ax.set_xlabel("ρ = π_θ / π_old"); ax.set_title(f"A = {A:+.0f}"); ax.legend()
plt.tight_layout(); plt.show()

print(f"{'A':>4} {'ρ':>5} | dL/dρ（数値微分）| 理論")
for A in [1.0, -1.0]:
    for rho in [0.5, 0.9, 1.0, 1.1, 1.5]:
        h = 1e-6
        d = (clip_obj(rho + h, A) - clip_obj(rho - h, A)) / (2 * h)
        clipped = (A > 0 and rho > 1 + EPS) or (A < 0 and rho < 1 - EPS)
        print(f"{A:>+4.0f} {rho:>5} | {d:>+16.3f} | {'0（クリップ）' if clipped else 'A'}")

**KL の推定**：PPO の実装では、更新のたびに新旧の方策がどれだけ離れたかを監視します。古い方策からのサンプルで $\rho = \pi_\theta/\pi_{\text{old}}$ を計算し、

$$
\widehat{\mathrm{KL}}_1 = \mathbb{E}\big[-\log\rho\big],\qquad \widehat{\mathrm{KL}}_3 = \mathbb{E}\big[(\rho - 1) - \log\rho\big]
$$

がよく使われます（どちらも $\mathrm{KL}(\pi_{\text{old}}\|\pi_\theta)$ の不偏推定。$\widehat{\mathrm{KL}}_3$ は各項が常に 0 以上で分散が小さい）。

In [ ]:
p_old = softmax(np.array([0.5, 0.0, -0.3, 0.2]))
p_new = softmax(np.array([0.9, -0.2, -0.6, 0.4]))
a = np.random.default_rng(2).choice(4, p=p_old, size=200000)
log_rho = np.log(p_new[a]) - np.log(p_old[a])
print(f"厳密な KL(π_old || π_new) = {np.sum(p_old * np.log(p_old / p_new)):.5f}")
print(f"KL_1 = E[-log ρ]          = {np.mean(-log_rho):.5f}（各項の標準偏差 {np.std(-log_rho):.3f}）")
print(f"KL_3 = E[(ρ-1) - log ρ]   = {np.mean(np.exp(log_rho) - 1 - log_rho):.5f}（各項の標準偏差 {np.std(np.exp(log_rho) - 1 - log_rho):.3f}）")

## 5. 小さな環境で PPO を動かす

第 3 節の一本道で、表形式の方策 $\theta[s,a]$ と価値関数 $V[s]$ を学習します。1 回の反復は次のとおりです。

1. 今の方策で 8 エピソード集める（各ステップの $s, a, r$ と $\log\pi_{\text{old}}(a\mid s)$ を記録）。
2. 今の $V$ を使って GAE（$\gamma = 0.95,\ \lambda = 0.95$）でアドバンテージを計算し、バッチ内で平均 0・標準偏差 1 に正規化する。価値関数の目標は $\hat{A}_t + V(s_t)$。
3. 同じデータで **$K$ エポック** 更新する。

次の 3 つを比べます（学習率は同じ）。

| 方法 | エポック数 $K$ | 方策の目的関数 |
|---|---|---|
| 方策勾配（PG） | 1 | $\rho\hat{A}$（$\rho=1$ なので通常の方策勾配と同じ） |
| クリップなし | 10 | $\rho\hat{A}$（代理目的関数をそのまま何度も最大化） |
| PPO | 10 | $\min(\rho\hat{A}, \mathrm{clip}(\rho)\hat{A})$ |

評価には、サンプルではなく **動的計画法で計算した厳密な期待収益** $J(\pi)$ を使います（最適値は常に右へ進む $\gamma^5\approx 0.774$）。乱数の種を変えて 10 回ずつ学習し、平均をとります。

In [ ]:
def collect(theta, n_episodes, rng):
    pi = softmax(theta)
    episodes = []
    for _ in range(n_episodes):
        s, ep = START, []
        for _ in range(200):
            a = rng.choice(2, p=pi[s])
            s2, r, done = env_step(s, a)
            ep.append((s, a, r, np.log(pi[s, a])))
            s = s2
            if done:
                break
        episodes.append(ep)
    return episodes


def ppo_iteration(theta, V, episodes, method, epochs, lr_pi=0.5, lr_v=0.5, eps=EPS):
    S, A_, ADV, RET, LOGP = [], [], [], [], []
    for ep in episodes:
        s_, a_, r_, lp_ = map(np.array, zip(*ep))
        adv = gae(r_, np.append(V[s_], 0.0), GAMMA, 0.95)
        S.append(s_); A_.append(a_); ADV.append(adv); RET.append(adv + V[s_]); LOGP.append(lp_)
    S, A_, ADV, RET, LOGP = map(np.concatenate, (S, A_, ADV, RET, LOGP))
    ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)
    n = len(S)
    pi_before = softmax(theta)
    clip_frac = 0.0
    for _ in range(epochs):
        pi = softmax(theta)
        rho = np.exp(np.log(pi[S, A_]) - LOGP)
        coef = ADV * rho                                         # 勾配の係数 Â ρ
        if method == "ppo":
            clipped = ((ADV > 0) & (rho > 1 + eps)) | ((ADV < 0) & (rho < 1 - eps))
            coef = np.where(clipped, 0.0, coef)
            clip_frac = clipped.mean()
        grad = np.zeros_like(theta)                              # Σ coef · (e_a - π(·|s))
        np.add.at(grad, (S, A_), coef)
        np.add.at(grad, S, -coef[:, None] * pi[S])
        theta = theta + lr_pi * grad / n                         # 勾配「上昇」
        gV = np.zeros_like(V)
        np.add.at(gV, S, V[S] - RET)
        V = V - lr_v * gV / n
    pi_after = softmax(theta)
    kl_step = np.mean(np.sum(pi_before[S] * np.log(pi_before[S] / pi_after[S]), axis=1))
    return theta, V, kl_step, clip_frac


def run(method, epochs, n_iters=60, seed=0):
    rng = np.random.default_rng(seed)
    theta, V = np.zeros((N_S, 2)), np.zeros(N_S)
    Js, kls = [J_of(theta)], []
    for _ in range(n_iters):
        theta, V, kl_step, _ = ppo_iteration(theta, V, collect(theta, 8, rng), method, epochs)
        Js.append(J_of(theta)); kls.append(kl_step)
    return np.array(Js), np.array(kls)


configs = {"PG (K=1)": ("pg", 1), "no clip (K=10)": ("noclip", 10), "PPO clip (K=10)": ("ppo", 10)}
seeds = range(10)
res = {name: [run(m, k, seed=s) for s in seeds] for name, (m, k) in configs.items()}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
for name, runs_ in res.items():
    Js = np.array([r_[0] for r_ in runs_]); kls = np.array([r_[1] for r_ in runs_])
    line, = axes[0].plot(Js.mean(0), label=name)
    axes[0].fill_between(range(Js.shape[1]), Js.min(0), Js.max(0), color=line.get_color(), alpha=0.12)
    axes[1].semilogy(kls.mean(0), label=name)
axes[0].axhline(GAMMA ** 5, color="gray", ls=":", label="optimal (always right)")
axes[0].axhline(exact_V(left_only)[START], color="gray", ls="--", lw=0.8, label="always left")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("exact J(π)")
axes[0].set_title("exact expected return (mean of 10 seeds, band = min..max)"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("iteration"); axes[1].set_title("KL(π_old || π_new) per iteration"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'':16s} | 60 反復後の J の平均 | 最小 | 最適に到達した割合 | 1 反復あたりの KL（平均）")
for name, runs_ in res.items():
    final = np.array([r_[0][-1] for r_ in runs_])
    kl_mean = np.mean([r_[1].mean() for r_ in runs_])
    print(f"{name:16s} | {final.mean():>18.3f} | {final.min():.3f} | {np.mean(final > 0.7):>16.0%} | {kl_mean:.4f}")

**観察**（乱数によって細部は変わります）：
- **PG（K=1）**：1 回集めたデータで 1 回しか更新しないので、改善がゆっくりです。
- **クリップなし（K=10）**：同じデータで代理目的関数を何度も最大化するので、1 反復あたりの方策の変化（KL）が大きくなります。第 3 節で見たとおり代理目的関数は方策が大きく動くと当てにならず、少ないサンプルのノイズにも過剰に反応するので、早い段階で偶然よく見えた行動に決め打ちしてしまい、**近くの小さな報酬（常に左）に収束してしまう** 試行が出ます。いったん方策が決定的になると、もう一方の行動を試さなくなるので抜け出せません。
- **PPO（K=10）**：同じ 10 エポックでも、$\rho$ が $1\pm\epsilon$ を超えた分の勾配が止まるので、1 反復あたりの変化が抑えられます。データを何度も使える利点を保ちつつ、安定して最適方策（常に右）に近づきます。

## 6. RLHF：GPT-2 small を PPO で学習する

RLHF では、言語モデルの生成を強化学習の問題として扱います（Ziegler et al., 2019; Ouyang et al., 2022）。

| 強化学習 | 言語モデル |
|---|---|
| 状態 $s_t$ | プロンプト $x$ とそれまでに生成したトークン $y_{<t}$ |
| 行動 $a_t$ | 次のトークン $y_t$ |
| 方策 $\pi_\theta(a_t\mid s_t)$ | 言語モデルの次トークン分布 |
| エピソード | 1 つの応答の生成 |

報酬は、応答全体に対する報酬モデルの点数 $R(x,y)$ を **最後のトークン** に与え、さらに **各トークンで参照方策との対数尤度比による罰則** を加えます。

$$
r_t = -\beta\,\log\frac{\pi_\theta(y_t\mid x, y_{<t})}{\pi_{\text{ref}}(y_t\mid x, y_{<t})} + \begin{cases} R(x, y) & (t = \text{最後}) \\ 0 & (\text{それ以外})\end{cases}
$$

1 エピソードの罰則の合計の期待値はちょうど $\beta\,\mathrm{KL}(\pi_\theta\|\pi_{\text{ref}})$ になるので、これは [dpo.ipynb](dpo.ipynb) の第 2 節で見た **KL 正則化付きの報酬最大化** を強化学習で解いていることになります。

このノートの設定：
- **方策**：GPT-2 small の各ブロックの `c_attn` に LoRA（$r=8$）を付けたもの。**参照方策** は LoRA を無効にした同じモデル。
- **価値関数**：最終層の隠れ状態に線形層（価値ヘッド）を付けて $V(s_t)$ を出す。
- **報酬モデルの代わり**：本物の報酬モデルの代わりに、[dpo.ipynb](dpo.ipynb) と同じ **単語リストによる感情スコア**（肯定語の数 − 否定語の数）を $R$ として使い、「映画レビューの書き出しから肯定的な続きを書く」ように学習します。
- アドバンテージは GAE（$\gamma = 1,\ \lambda = 0.95$）、PPO は $\epsilon = 0.2$、1 反復あたり 4 エポック。

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
REPO = "openai-community/gpt2"
tok = AutoTokenizer.from_pretrained(REPO)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(REPO).to(device).eval()    # ドロップアウトは無効
for p in model.parameters():
    p.requires_grad_(False)


class LoRALayer(nn.Module):
    """GPT-2 の Conv1D（y = xW + b, W: (in, out)）に LoRA を足す: y = base(x) + (α/r)(xA)B"""

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        d_in, d_out = base.weight.shape
        self.base, self.scaling, self.enabled = base, alpha / r, True
        self.A = nn.Parameter(torch.empty(d_in, r).uniform_(-1 / math.sqrt(d_in), 1 / math.sqrt(d_in)))
        self.B = nn.Parameter(torch.zeros(r, d_out))

    def forward(self, x):
        y = self.base(x)
        return y + self.scaling * ((x @ self.A) @ self.B) if self.enabled else y


torch.manual_seed(0)
lora_layers = []
for block in model.transformer.h:
    block.attn.c_attn = LoRALayer(block.attn.c_attn).to(device)
    lora_layers.append(block.attn.c_attn)
value_head = nn.Linear(model.config.n_embd, 1).to(device)
nn.init.zeros_(value_head.weight); nn.init.zeros_(value_head.bias)
policy_params = [p for p in model.parameters() if p.requires_grad]
print(f"device = {device}, 方策の学習パラメータ {sum(p.numel() for p in policy_params):,}, "
      f"価値ヘッド {sum(p.numel() for p in value_head.parameters()):,}")


@contextmanager
def reference_model():
    """LoRA を無効にした状態 = 参照方策 π_ref（抜けるときは元の状態に戻す）"""
    prev = [l.enabled for l in lora_layers]
    for l in lora_layers:
        l.enabled = False
    try:
        yield model
    finally:
        for l, e in zip(lora_layers, prev):
            l.enabled = e


# トークン数（先頭の <|endoftext|> を含めて 4）をそろえておくと、16 本を 1 回の generate でまとめて生成でき、パディングも要らない
prompts = ["The movie was", "This film is", "The acting was", "The story was",
           "The ending was", "The soundtrack was", "The characters were", "The music was"]
prompt_ids = tok([tok.eos_token + p for p in prompts], return_tensors="pt").input_ids.to(device)
assert prompt_ids.shape == (len(prompts), 4)
POS = {"good", "great", "excellent", "wonderful", "amazing", "loved", "love", "beautiful", "best", "fantastic",
       "enjoyed", "brilliant", "fun", "perfect", "masterpiece", "superb", "delightful", "gorgeous", "stunning",
       "impressive", "inspiring", "clever", "gripping", "happy", "favorite", "awesome", "nice", "funny"}
NEG = {"bad", "terrible", "awful", "boring", "worst", "hated", "hate", "poor", "waste", "dull", "disappointing",
       "horrible", "stupid", "mess", "annoying", "lazy", "cheap", "confusing", "bored", "regretted", "ruined",
       "forgettable", "wooden", "painfully", "worse", "sad", "ugly", "fails"}


def sentiment(text):
    words = re.findall(r"[a-z']+", text.lower())
    return sum(w in POS for w in words) - sum(w in NEG for w in words)

### 6.1 ロールアウト・報酬・GAE

- **ロールアウト**：各プロンプトから 2 本ずつ（計 16 本）、方策から **そのまま** サンプリングして最大 20 トークン生成します（top-k などで分布を変えると、記録した $\log\pi_{\text{old}}$ と実際のサンプリング分布がずれるため）。`<|endoftext|>` が出たら、そこまで（`<|endoftext|>` 自身を含む）を 1 エピソードとします。
- 1 回の順伝播で、各トークンの $\log\pi_\theta(y_t\mid s_t)$ と価値 $V(s_t)$ を求めます。位置 $p$ のトークンの確率は位置 $p-1$ の出力から決まるので、行動 $y_t$ に対応する価値は「そのトークンを選ぶ直前の状態」の隠れ状態から計算します。

In [ ]:
GEN_LEN, BETA, GAMMA_LM, LAM_LM = 20, 0.05, 1.0, 0.95


@torch.no_grad()
def generate_from_prompts(m, p_ids, n_per_prompt):
    """方策から（分布を変えずに）サンプリング。act[:, p-1] = 1 ⇔ 位置 p のトークンが行動"""
    ids = p_ids.repeat_interleave(n_per_prompt, dim=0)
    gen = m.generate(ids, attention_mask=torch.ones_like(ids), do_sample=True, top_k=0, top_p=1.0,
                     max_new_tokens=GEN_LEN, pad_token_id=tok.eos_token_id)
    P = ids.shape[1]
    resp = gen[:, P:]
    is_eos = (resp == tok.eos_token_id).long()
    act_resp = ((is_eos.cumsum(1) - is_eos) == 0).float()         # 最初の EOS まで（EOS を含む）
    act = torch.zeros(gen.shape[0], gen.shape[1] - 1, device=device)
    act[:, P - 1:] = act_resp
    texts = [tok.decode(r[a.bool()], skip_special_tokens=True) for r, a in zip(resp, act_resp)]
    return gen, torch.ones_like(gen), act, texts


def rollout(n_per_prompt=2):
    return generate_from_prompts(model, prompt_ids, n_per_prompt)


def logprobs_values(m, ids, attn):
    """logp[:, t] = log π(ids[:, t+1] | ids[:, :t+1]),  values[:, t] = V(その直前の状態)"""
    out = m(input_ids=ids, attention_mask=attn, output_hidden_states=True)
    logp = torch.log_softmax(out.logits[:, :-1].float(), dim=-1).gather(2, ids[:, 1:, None])[..., 0]
    values = value_head(out.hidden_states[-1][:, :-1].detach().float()).squeeze(-1)   # 価値ヘッドだけを学習
    return logp, values


def compute_gae(rewards, values, act):
    """行動の位置だけを使って後ろから GAE を計算（行動の後ろは終端 → 価値 0）"""
    T = rewards.shape[1]
    adv = torch.zeros_like(rewards)
    last = torch.zeros(rewards.shape[0], device=rewards.device)
    for t in reversed(range(T)):
        nxt = act[:, t + 1] if t + 1 < T else torch.zeros_like(last)
        next_v = values[:, t + 1] * nxt if t + 1 < T else torch.zeros_like(last)
        delta = rewards[:, t] + GAMMA_LM * next_v - values[:, t]
        last = delta + GAMMA_LM * LAM_LM * nxt * last
        adv[:, t] = last * act[:, t]
    return adv


def masked_mean(x, m):
    return (x * m).sum() / m.sum()


ids, attn, act, texts = rollout()
with torch.no_grad():
    logp_old, values_old = logprobs_values(model, ids, attn)
    with reference_model() as ref:
        logp_ref, _ = logprobs_values(ref, ids, attn)
print("ロールアウトの形状:", tuple(ids.shape), " 行動トークン数:", int(act.sum()))
print("学習開始時（π_θ = π_ref）の max|log π_θ - log π_ref| =", (logp_old - logp_ref).abs().max().item())
print("例:", [repr(t) for t in texts[:3]])

### 6.2 PPO の学習ループ

1 反復ごとに、ロールアウト → 報酬（最後のトークンに感情スコア、各トークンに $-\beta\log(\pi_{\text{old}}/\pi_{\text{ref}})$）→ GAE → 4 エポックの更新、を行います。損失は

$$
\mathcal{L} = -\underbrace{\mathbb{E}_t\Big[\min\big(\rho_t\hat{A}_t,\ \mathrm{clip}(\rho_t, 1\pm\epsilon)\hat{A}_t\big)\Big]}_{\text{方策}} + c_v\,\underbrace{\tfrac{1}{2}\,\mathbb{E}_t\Big[\big(V_\phi(s_t) - \hat{R}_t\big)^2\Big]}_{\text{価値関数}},\qquad \hat{R}_t = \hat{A}_t + V_{\text{old}}(s_t)
$$

です。最初のエポックでは方策がまだ変わっていないので $\rho_t = 1$ になるはずで、それも確認します。方策の学習率は $10^{-3}$、25 反復学習し、20 反復目の LoRA の重みも保存しておきます（第 6.3 節で最終状態と比べる）。

In [ ]:
opt = torch.optim.Adam([{"params": policy_params, "lr": 1e-3}, {"params": value_head.parameters(), "lr": 1e-2}])
N_ITERS, PPO_EPOCHS, VF_COEF, SNAPSHOT_ITER = 25, 4, 0.5, 20
history = []
t0 = time.perf_counter()
for it in range(1, N_ITERS + 1):
    ids, attn, act, texts = rollout()
    scores = torch.tensor([float(sentiment(t)) for t in texts], device=device)
    with torch.no_grad():
        logp_old, values_old = logprobs_values(model, ids, attn)
        with reference_model() as ref:
            logp_ref, _ = logprobs_values(ref, ids, attn)
        log_ratio_ref = (logp_old - logp_ref) * act                     # 各トークンの log(π/π_ref)
        rewards = -BETA * log_ratio_ref
        last_idx = act.shape[1] - 1 - act.flip(1).argmax(1)              # 各系列の最後の行動の位置
        rewards[torch.arange(len(texts)), last_idx] += scores
        adv = compute_gae(rewards, values_old, act)
        returns = adv + values_old
        adv_n = (adv - masked_mean(adv, act)) / (torch.sqrt(masked_mean((adv - masked_mean(adv, act)) ** 2, act)) + 1e-8)

    for epoch in range(PPO_EPOCHS):
        logp, values = logprobs_values(model, ids, attn)
        ratio = torch.exp(logp - logp_old)
        if it == 1 and epoch == 0:
            first_ratio_err = (ratio - 1).abs().mul(act).max().item()
        pg_loss = -masked_mean(torch.min(ratio * adv_n, torch.clamp(ratio, 1 - EPS, 1 + EPS) * adv_n), act)
        v_loss = 0.5 * masked_mean((values - returns) ** 2, act)
        loss = pg_loss + VF_COEF * v_loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    with torch.no_grad():
        clip_frac = masked_mean(((ratio - 1).abs() > EPS).float(), act).item()
        approx_kl = masked_mean((ratio - 1) - torch.log(ratio), act).item()     # KL_3（最後のエポック）
    history.append(dict(score=scores.mean().item(), kl_ref=log_ratio_ref.sum(1).mean().item(),
                        clip=clip_frac, kl_step=approx_kl, v_loss=v_loss.item()))
    if it == SNAPSHOT_ITER:
        snapshot = [(l.A.detach().clone(), l.B.detach().clone()) for l in lora_layers]
    if it == 1 or it % 5 == 0:
        h = history[-1]
        print(f"iter {it:3d}: 感情スコア {h['score']:+.2f}  KL(π||π_ref) {h['kl_ref']:5.2f}  "
              f"clip 率 {h['clip']:.2f}  KL(old→new) {h['kl_step']:.4f}  価値損失 {h['v_loss']:.3f}   例: {texts[0]!r}")
print(f"\n学習時間: {time.perf_counter() - t0:.0f} 秒")
print(f"最初の反復・最初のエポックの max|ρ - 1| = {first_ratio_err:.1e}（方策がまだ変わっていないので ρ = 1）")
assert first_ratio_err < 1e-4

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
its = range(1, len(history) + 1)
axes[0].plot(its, [h["score"] for h in history]); axes[0].set_title("reward (sentiment score of samples)")
axes[1].plot(its, [h["kl_ref"] for h in history]); axes[1].set_title("KL(π || π_ref) per response [nats]")
axes[2].plot(its, [h["clip"] for h in history]); axes[2].set_title("clip fraction (last epoch)")
axes[3].plot(its, [h["kl_step"] for h in history]); axes[3].set_title("KL(π_old || π_new) per iteration")
for ax in axes:
    ax.axvline(SNAPSHOT_ITER, color="tab:red", ls="--", lw=1)
    ax.set_xlabel("iteration")
plt.tight_layout(); plt.show()

### 6.3 学習前・20 反復後・25 反復後の生成を比べる

感情スコアに加えて、文章の自然さのごく粗い目安として **異なり語率**（異なる単語の数 ÷ 全単語数。同じ単語の繰り返しが多いと小さくなる）を見ます。

In [ ]:
@contextmanager
def adapter_weights(snap):
    """一時的に LoRA の重みを snap に差し替える"""
    current = [(l.A.detach().clone(), l.B.detach().clone()) for l in lora_layers]
    with torch.no_grad():
        for l, (A, B) in zip(lora_layers, snap):
            l.A.copy_(A); l.B.copy_(B)
    try:
        yield model
    finally:
        with torch.no_grad():
            for l, (A, B) in zip(lora_layers, current):
                l.A.copy_(A); l.B.copy_(B)


def samples(m, n=4, seed=0):
    torch.manual_seed(seed)
    _, _, _, texts = generate_from_prompts(m, prompt_ids[:4], n)
    return [(prompts[k // n], t) for k, t in enumerate(texts)]


def distinct_ratio(text):
    words = re.findall(r"[a-z']+", text.lower())
    return len(set(words)) / max(1, len(words))


outs = {}
with reference_model() as ref:
    outs["学習前 (π_ref)"] = samples(ref)
with adapter_weights(snapshot) as m20:
    outs[f"{SNAPSHOT_ITER} 反復後"] = samples(m20)
outs[f"{N_ITERS} 反復後"] = samples(model)

print(f"{'':14s} | 平均感情スコア | 肯定的 | 否定的 | 異なり語率")
for name, ss in outs.items():
    sc = np.array([sentiment(t) for _, t in ss])
    dr = np.mean([distinct_ratio(t) for _, t in ss])
    print(f"{name:14s} | {sc.mean():+14.2f} | {np.mean(sc > 0):6.0%} | {np.mean(sc < 0):6.0%} | {dr:10.2f}")
print()
for k in range(0, 16, 4):
    print(f"[{prompts[k // 4]}]")
    for name, ss in outs.items():
        print(f"   {name:14s}: {ss[k][1]!r}")

**観察**（数値や崩れ始める反復は、乱数や実行環境によって変わります）：
- 報酬（感情スコア）は反復とともに上がり、生成される続きが肯定的になっていきます。DPO と違い、PPO は **学習中に自分で文章を生成し、その点数から学ぶ** ので、好みデータを用意する代わりに報酬を返す仕組みが必要です。
- 最初の反復の最初のエポックでは $\rho = 1$ でした。その後は clip 率と 1 反復あたりの KL が、方策の 1 回の更新の大きさを示しています。報酬が急に上がり始めるとクリップされる割合も増え、PPO が更新を抑えようとしているのがわかります。
- **reward hacking**：ここでの報酬は単語リストなので、「肯定語を並べるだけ」で点数が上がります。学習の後半では、報酬は急上昇する一方で、肯定語の羅列のような文章になり、異なり語率が下がり、参照方策との KL も急増します。PPO は与えられた報酬を忠実に最大化するので、**報酬の弱点をそのまま突きます**。実際の RLHF で報酬モデルの弱点が突かれるのも同じ構図です。
- KL 罰則の係数 $\beta$ を大きくすると、参照方策から離れることのコストが上がり、このような崩壊は起こりにくくなります（そのぶん報酬の伸びは遅くなる）。[dpo.ipynb](dpo.ipynb) の第 2 節で見た、$\beta$ による「報酬」と「参照方策からの距離」のトレードオフそのものです。実際の RLHF では、$\beta$ の調整、報酬モデルの改良、学習の早期停止などで対処します。

## 7. まとめと参考文献

### このノートで確認したこと

| 節 | 確認したこと |
|---|---|
| 1 | REINFORCE の勾配推定は不偏。ベースラインを引いても平均は変わらず、分散が大きく下がる |
| 2 | GAE の漸化式と定義の一致、$\lambda=0$ で TD 誤差、$\lambda=1$ で $G_t - V(s_t)$ |
| 3 | 方策勾配定理と性能差分補題を厳密計算で検証。代理目的関数は方策の近くでは正確だが、離れるほど（KL が大きいほど）真の改善との誤差が大きくなる |
| 4 | クリップ目的関数の形と勾配（$\rho$ が $1\pm\epsilon$ を越えた側で勾配 0、悲観的な下界）、KL 推定量 |
| 5 | 同じデータを 10 エポック使うとき、クリップなしは方策の変化が大きく局所解に陥る試行が出る。PPO は安定して最適方策に近づく |
| 6 | RLHF の定式化（最後のトークンに報酬、各トークンに KL 罰則）で GPT-2 small を PPO で学習。最初のエポックで $\rho=1$、報酬と KL の推移、clip 率。単語リストの報酬は肯定語の羅列で突かれる（reward hacking） |

### 参考文献
- J. Schulman, F. Wolski, P. Dhariwal, A. Radford, O. Klimov. "Proximal Policy Optimization Algorithms." 2017. [arXiv:1707.06347](https://arxiv.org/abs/1707.06347)
- J. Schulman, P. Moritz, S. Levine, M. Jordan, P. Abbeel. "High-Dimensional Continuous Control Using Generalized Advantage Estimation." ICLR 2016. [arXiv:1506.02438](https://arxiv.org/abs/1506.02438)
- 本文中で触れた研究：Williams (1992)（REINFORCE）、Kakade & Langford (2002)（性能差分補題・保守的な方策反復）、Schulman et al. (2015) *Trust Region Policy Optimization*（TRPO）、Ziegler et al. (2019) *Fine-Tuning Language Models from Human Preferences*、Ouyang et al. (2022) *Training language models to follow instructions with human feedback*（InstructGPT）